# End-to-End DBSCAN Clustering & Outlier Detection (No PCA)
This notebook implements **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** directly across all **8 original features** of the California Housing dataset (`20,640` records) without any dimensionality reduction (no PCA).

### What This Pipeline Demonstrates:
1. **Dataset Loading & Pre-Clustering EDA**: Inspecting raw distributions, scale differences, and extreme anomalies before training.
2. **Feature Standardization & Optimal $\varepsilon$ Discovery**: Scaling all 8 features equally and finding the mathematical elbow point using the $k$-Distance graph.
3. **Multi-Feature DBSCAN Clustering**: Grouping all 8 features simultaneously into dense spatial/economic clusters (`Cluster >= 0`) while isolating anomalies as Noise (`Cluster = -1`).
4. **Clustering Evaluation & Accuracy Scores**: Computing internal validation scores (Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Score) and comparing how DBSCAN handles single vs. multiple cluster regimes across $\varepsilon$ thresholds.
5. **Post-Clustering EDA & Direct Feature Visualizations**: Examining how outliers are captured directly in the original feature space (Geospatial coordinates, Median Income, Average Rooms, Population, and House Value).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Set clean visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

---
### 1. Load Dataset (20,640 Records)

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print(f"Total Records : {df.shape[0]:,}")
print(f"Total Columns : {df.shape[1]}")
df.head()

---
### 2. Pre-Clustering EDA (Before DBSCAN Training)

In [ ]:
print("[PRE-EDA] Summary Statistics (Raw Original Features):")
print(df.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].round(3))

print("\n[PRE-EDA] Missing Values Check:")
missing_count = df.isnull().sum().sum()
print(f"Total Missing Values: {missing_count}")

In [ ]:
# Visualize Raw Distributions & Unclustered Outliers Before Training
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['MedInc'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title("Pre-EDA: Median Income (Right-Skewed)")

sns.histplot(df['AveRooms'], bins=50, kde=True, ax=axes[1], color='coral')
axes[1].set_xlim(0, 20)
axes[1].set_title("Pre-EDA: Average Rooms (Extreme Outliers > 20)")

sns.scatterplot(
    data=df, x='Longitude', y='Latitude', hue='MedHouseVal',
    palette='coolwarm', alpha=0.5, s=15, ax=axes[2]
)
axes[2].set_title("Pre-EDA: Raw Geospatial Distribution (Unclustered)")

plt.tight_layout()
plt.show()

---
### 3. Feature Scaling & Optimal Epsilon ($\varepsilon$) Discovery

In [ ]:
# All 8 features used directly (no PCA or dimensionality reduction)
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
X = df[features].values

# Standardize features so Euclidean distance treats all 8 dimensions equally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Rule of thumb: MinPts = 2 * number of features = 16
min_pts = 2 * len(features)

# Compute distances to the k-th nearest neighbor
neighbors = NearestNeighbors(n_neighbors=min_pts, n_jobs=-1)
distances, _ = neighbors.fit(X_scaled).kneighbors(X_scaled)
k_distances = np.sort(distances[:, min_pts - 1])

# Plot k-Distance Elbow Curve
plt.figure(figsize=(10, 4))
plt.plot(k_distances, color='darkblue', linewidth=2)
plt.axhline(y=1.15, color='red', linestyle='--', label='Multi-Cluster Threshold (ε = 1.15)')
plt.axhline(y=2.20, color='green', linestyle=':', label='Single-Dominant Cluster Threshold (ε = 2.20)')
plt.title(f"k-Distance Graph (k = MinPts = {min_pts}) Across All 8 Scaled Features")
plt.xlabel("Data Points Sorted by Distance")
plt.ylabel(f"{min_pts}-th Nearest Neighbor Euclidean Distance")
plt.ylim(0, 5)
plt.legend()
plt.tight_layout()
plt.show()

---
### 4. How DBSCAN Converts All 8 Features into Single vs. Multiple Clusters & Captures Outliers

In [ ]:
# Demonstrate how ε controls merging all 8 features into a single cluster vs. distinct clusters
print("COMPARING DBSCAN REGIMES ACROSS ALL 8 FEATURES (MinPts = 16):")
print("-" * 75)
for eps_test in [0.90, 1.15, 1.50, 2.20]:
    labels_test = DBSCAN(eps=eps_test, min_samples=min_pts, n_jobs=-1).fit_predict(X_scaled)
    n_clust = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    n_outliers = (labels_test == -1).sum()
    outlier_pct = (n_outliers / len(df)) * 100
    regime = "Single Dense Core" if n_clust == 1 else f"Multiple Clusters ({n_clust})"
    print(f"ε = {eps_test:.2f} | {regime:<25} | Outliers Captured: {n_outliers:>5,} ({outlier_pct:.2f}%)")

# Fit Final DBSCAN Model at Optimal Elbow (ε = 1.15)
eps_optimal = 1.15
dbscan = DBSCAN(eps=eps_optimal, min_samples=min_pts, metric='euclidean', n_jobs=-1)
df['Cluster'] = dbscan.fit_predict(X_scaled)
df['Point_Type'] = np.where(df['Cluster'] == -1, 'Outlier / Noise (-1)', 'Clustered Inlier (>= 0)')

---
### 5. Clustering Validation & Accuracy Metrics

In [ ]:
n_clusters = len(set(df['Cluster'])) - (1 if -1 in df['Cluster'].values else 0)
n_noise = (df['Cluster'] == -1).sum()
n_inliers = len(df) - n_noise

print("=" * 65)
print(f"FINAL DBSCAN MODEL PERFORMANCE (ε = {eps_optimal}, MinPts = {min_pts})")
print("=" * 65)
print(f"Total Clusters Discovered   : {n_clusters}")
print(f"Clustered Inlier Points     : {n_inliers:,} ({n_inliers / len(df) * 100:.2f}%)")
print(f"Outliers / Noise (-1)       : {n_noise:,} ({n_noise / len(df) * 100:.2f}%)")

# Evaluate internal clustering quality scores on clustered points
mask_valid = df['Cluster'] != -1
if n_clusters > 1:
    sil_score = silhouette_score(X_scaled[mask_valid], df.loc[mask_valid, 'Cluster'])
    db_index = davies_bouldin_score(X_scaled[mask_valid], df.loc[mask_valid, 'Cluster'])
    ch_score = calinski_harabasz_score(X_scaled[mask_valid], df.loc[mask_valid, 'Cluster'])
    print("-" * 65)
    print(f"Silhouette Score            : {sil_score:.4f}  (Range: [-1, 1], Higher = Better separation)")
    print(f"Davies-Bouldin Index        : {db_index:.4f}  (Min: 0, Lower = Tighter clusters)")
    print(f"Calinski-Harabasz Score     : {ch_score:.2f}  (Higher = Better defined clusters)")
else:
    print("Single cluster formed; multi-cluster separation metrics require >= 2 clusters.")

---
### 6. Post-Clustering EDA (Measuring Outlier Capture & Feature Changes)

In [ ]:
print("[POST-EDA] Record Count per Cluster ID (-1 = Noise/Outliers):")
print(df['Cluster'].value_counts().rename(index={-1: 'Noise (-1)'}))

print("\n[POST-EDA] Before vs. After Outlier Removal (Mean & Standard Deviation):")
comparison_df = pd.DataFrame({
    'Before_Mean (All Data)': df[features].mean(),
    'After_Mean (Inliers)': df[mask_valid][features].mean(),
    'Outlier_Mean (Noise -1)': df[~mask_valid][features].mean(),
    'Before_Std (All Data)': df[features].std(),
    'After_Std (Inliers)': df[mask_valid][features].std()
})
print(comparison_df.round(3))

print("\n[POST-EDA] Mean Feature Profile by Cluster:")
print(df.groupby('Cluster')[features + ['MedHouseVal']].mean().round(3))

---
### 7. Post-Training Visualizations (Direct Original Features — Zero PCA)

In [ ]:
# Build dynamic color mapping so Noise (-1) is always distinct grey
unique_clusters = sorted(df['Cluster'].unique())
n_valid_clusters = len([c for c in unique_clusters if c != -1])
palette_colors = sns.color_palette("tab10", n_colors=max(1, n_valid_clusters))

cluster_palette = {}
color_idx = 0
for c in unique_clusters:
    if c == -1:
        cluster_palette[c] = "#7f7f7f"  # Grey for Noise/Outliers
    else:
        cluster_palette[c] = palette_colors[color_idx % len(palette_colors)]
        color_idx += 1

point_palette = {
    'Clustered Inlier (>= 0)': '#1f77b4',
    'Outlier / Noise (-1)': '#d62728'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Geospatial Cluster Assignment (Original Longitude vs. Latitude)
sns.scatterplot(
    data=df, x='Longitude', y='Latitude', hue='Cluster',
    palette=cluster_palette, alpha=0.65, s=16, ax=axes[0, 0]
)
axes[0, 0].set_title("1. Geospatial Clusters vs. Outliers (Original Coordinates)")

# Plot 2: Structural & Economic Outlier Isolation (MedInc vs. AveRooms)
sns.scatterplot(
    data=df, x='MedInc', y='AveRooms', hue='Point_Type',
    palette=point_palette, alpha=0.65, s=18, ax=axes[0, 1]
)
axes[0, 1].set_ylim(0, 25)
axes[0, 1].set_title("2. Outlier Capture in Original Features (MedInc vs. AveRooms)")

# Plot 3: House Value Distribution per Cluster (Version-Safe Boxplot)
sns.boxplot(
    data=df, x='Cluster', y='MedHouseVal', hue='Cluster',
    palette=cluster_palette, dodge=False, ax=axes[1, 0]
)
if axes[1, 0].get_legend() is not None:
    axes[1, 0].get_legend().remove()
axes[1, 0].set_title("3. Median House Value Distribution Across Clusters & Noise (-1)")

# Plot 4: Variance Reduction in AveRooms (Inliers vs. Captured Outliers)
sns.boxplot(
    data=df, x='Point_Type', y='AveRooms', hue='Point_Type',
    palette=point_palette, dodge=False, ax=axes[1, 1]
)
if axes[1, 1].get_legend() is not None:
    axes[1, 1].get_legend().remove()
axes[1, 1].set_ylim(0, 15)
axes[1, 1].set_title("4. Outlier Isolation Effect on 'AveRooms' Variance")

plt.tight_layout()
plt.show()